## Linux `locate` Command
[LabEx article](https://labex.io/tutorials/linux-linux-locate-command-with-practical-examples-422765)

Used to quickly search for files and directories on your system. The locate command uses a pre-built database to perform fast searches, making it more efficient than the find command, especially for large file systems.

> **Support rule to keep:** `locate` = fast but can lie (stale database); `find` = slow but always true. When a support ticket says "the file exists but the tool can't find it," suspect index/cache staleness before assuming the tool is broken — same pattern applies later to DNS caches, `apt` package lists, and browser caches.

### Understand the Purpose and Usage of the locate Command

The `locate` command searches the database for the specified file or directory name and returns the matching results.

To use the `locate` command, you first need to install the `mlocate` package, which provides the `locate` command and maintains the database of file locations.

In [ ]:
dennie@HP:~/linux-tut$ sudo apt-get update && sudo apt-get install -y plocate
Hit:1 http://archive.ubuntu.com/ubuntunoble InRelease
Hit:2 http://archive.ubuntu.com/ubuntunoble-updates InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntunoble-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
plocate is already the newest version (1.1.19-2ubuntu2).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.

The `apt-get update` command updates the package lists, and the `apt-get install` command installs the `mlocate` package.

(`plocate` provides the same `locate`/`updatedb` commands as `mlocate` — faster index, same usage.) Requires sudo.

Verify that the `locate` command is available by running the following command:

In [ ]:
dennie@HP:~/linux-tut$ locate --version
plocate 1.1.19
Copyright 2020 Steinar H. Gunderson
License GPLv2+: GNU GPL version 2 or later <https://gnu.org/licenses/gpl.html>.
This is free software: you are free tochange and redistribute it.
There is NO WARRANTY, to the extent permitted by law.

In [ ]:
dennie@HP:~/linux-tut$ locate PROGRESS.md
/home/dennie/linux-tut/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Algorighms & Data Structures/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Blackout Emergency Folders/Algorighms & DataStructures/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Obsidian vault/Trading/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/trading-lab/PROGRESS.md
/mnt/d/python_practice/Algorighms & Data Structures/PROGRESS.md

This will display all the locations on your system where a file or directory named "PROGRESS.md" is found.

You can also use wildcards with the `locate` command to perform more complex searches. For example, to search for all files and directories that start with "PROGRESS", you can run:

In [ ]:
dennie@HP:~/linux-tut$ locate PROGRESS*
/home/dennie/linux-tut/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Algorighms & Data Structures/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Blackout Emergency Folders/Algorighms & Data Structures/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Obsidian vault/Trading/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/trading-lab/PROGRESS.md
/mnt/d/python_practice/Algorighms & Data Structures/PROGRESS.md

### Search for Files and Directories Using the `locate` Command

In [ ]:
cd labex
mkdir sample_dir
touch sample_file.txt

#### Build or Refresh the database manually

In [ ]:
dennie@HP:~/linux-tut$ locate sample_dir
dennie@HP:~/linux-tut$ locate sample_file.txt

NO files or directory is found, means that database doesn't know about them.

PURPOSE (support): `locate` searches a prebuilt database, not the live filesystem. That's the #1 gotcha — and exactly what's biting you now.

**Diagnosis path:**

1. Check when the database was last built:

In [ ]:
dennie@HP:~/linux-tut$ ls -la /var/lib/plocate/plocate.db
-rw-r----- 1 root plocate 24394978 Sep  5 07:30 /var/lib/plocate/plocate.db

If it doesn't exist, or its timestamp is _before_ you created `sample_dir`/`sample_file.txt`, that's your answer — the database has never been built, or was built before your files existed.

2. Build/refresh the database manually (requires sudo — it needs to read across the whole filesystem):

In [ ]:
dennie@HP:~/linux-tut$ sudo updatedb
[sudo] password for dennie: 

3. Now retry:

In [ ]:
dennie@HP:~/linux-tut$ locate sample_file.txt
/home/dennie/linux-tut/labex/sample_file.txt

dennie@HP:~/linux-tut$ locate sample_dir
/home/dennie/linux-tut/labex/sample_dir

Fixed — `updatedb` refreshed the database and `locate` now finds both.

**WHAT TO LOOK FOR:** `updatedb` runs silently and takes a few seconds to a minute. No output = success.

If it errors about permissions or paths under `/mnt/c`, that's WSL noise you can ignore (Windows-mounted drives often throw permission warnings during indexing).

---

**TICKET SCENARIO:** "I just created a file and `locate` says it doesn't exist, but `ls` shows it right there." → classic **stale-index issue**.

> This is the difference between `locate` (fast, database-backed, can lag reality) and `find` (slow, walks the live filesystem, always accurate) — a distinction worth having ready for an interview.

**GOTCHA:** On most standard Linux distros, `updatedb` runs automatically once a day via cron. **WSL-specific:** cron often isn't running by default in WSL, so the daily auto-refresh silently never happens unless you start it:

In [ ]:
dennie@HP:~/linux-tut$ sudo service cron start
[sudo] password for dennie: 

Until you do that (or run `sudo updatedb` manually), your `locate` database can go stale indefinitely — worth checking `service cron status` on any WSL box before trusting `locate` results in a real investigation.

**TRY NOW:**

In [ ]:
sudo updatedb
locate sample_dir

One more WSL-specific thing worth locking in now: check whether cron is actually running, since that's what would've prevented this staleness automatically going forward.

In [ ]:
dennie@HP:~/linux-tut$ service cron status
● cron.service - Regular background program proce>
     Loaded: loaded (/usr/lib/systemd/system/cron>
     Active: active (running) since Sat 2026-09-1>
       Docs: man:cron(8)
   Main PID: 156 (cron)
      Tasks: 1 (limit: 9427)
     Memory: 428.0K (peak: 1.7M)
        CPU: 17ms
     CGroup: /system.slice/cron.service
             └─156 /usr/sbin/cron -f -P

Sep 12 07:35:04 HP systemd[1]: Started cron.servi>
Sep 12 07:35:04 HP (cron)[156]: cron.service: Ref>
Sep 12 07:35:04 HP cron[156]: (CRON) INFO (pidfil>
Sep 12 07:35:04 HP cron[156]: (CRON) INFO (Runnin>
lines 1-15/15 (END)

Good news — cron is already running on your system (started 10 min ago at boot, `active (running)`), so `updatedb` will run automatically on its normal daily schedule via `/etc/cron.daily/plocate` from now on. No manual `sudo service cron start` needed here — that was a "just in case" check, not an actual problem this time.

**WHAT TO LOOK FOR** in that output: `Active: active (running)` is the healthy state. If you ever see `inactive (dead)` or `failed` here, that's when you'd need `sudo service cron start` (WSL doesn't always auto-start cron on distro launch the way bare-metal Linux does).

Solid diagnosis path today: stale index → confirmed via `ls -la` on the db → fixed with `sudo updatedb` → verified root cause won't recur by checking cron health. That's a full support loop.

The `locate` command is **case-sensitive by default**, but you can make it **case-insensitive** by using the `-i` option:

In [ ]:
dennie@HP:~/linux-tut$ locate -i progress.md
/home/dennie/linux-tut/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Algorighms & Data Structures/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Blackout Emergency Folders/Algorighms & Data Structures/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/Obsidian vault/Trading/PROGRESS.md
/mnt/c/Users/dennie/OneDrive/trading-lab/PROGRESS.md
/mnt/d/python_practice/Algorighms & Data Structures/PROGRESS.md